# 🔬 Krypto: Mobile Forensic Activity Reconstruction Fine-Tuning
### Fine-Tuning Lightweight LLMs (Gemma-2-2B / Gemma-4-E2B) with Unsloth on H100/A100 GPU

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/emmanuelbadmus/Krypto/blob/main/FineTune_Gemma_Forensics.ipynb)

---
### 🎯 Objectives
1. **100% Citation Discipline**: Enforce strict `[EVT-xxxxxxxxxxxx]` ground-truth citations to eliminate hallucinations.
2. **Absence & SQLCipher Auditing**: Teach the model to explicitly document unrecoverable/encrypted data (Signal, Google Podcasts, WeChat residue).
3. **Multi-Signal Indirect Commute Deduction**: Synthesize unlogged Android Auto commutes from Bluetooth and charging logs.
4. **Beat Non-AI Baseline**: Outperform the 68.2% baseline recall in `baseline_benchmark_report.txt`.

## 🛠️ Step 1: Install Unsloth & Dependencies

In [ ]:
# Install Unsloth for 2x faster training and 70% lower VRAM usage
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install torch transformers datasets trl peft accelerate bitsandbytes sentencepiece protobuf scikit-learn tabulate

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
    print(f"bfloat16 Supported: {torch.cuda.is_bf16_supported()}")

## 📂 Step 2: Load Training & Validation Datasets
*(Automatically searches locally, handles private repo auth, or prompts a 1-click upload in Colab)*

In [ ]:
import os
import sys

def find_file(filename):
    for root, dirs, files in os.walk("."):
        if filename in files:
            return os.path.abspath(os.path.join(root, filename))
    return None

train_file = find_file("train.jsonl")
val_file = find_file("val.jsonl")

# If not found, prompt 1-click upload in Colab
if not train_file:
    print("[Notice] 'train.jsonl' not found in current directory.")
    try:
        from google.colab import files
        print("\n👉 Please upload 'train.jsonl' and 'val.jsonl' (from your local Krypto/data/splits/ folder):")
        uploaded = files.upload()
        os.makedirs("data/splits", exist_ok=True)
        for fname in uploaded:
            dest = os.path.join("data/splits", fname)
            with open(dest, "wb") as f:
                f.write(uploaded[fname])
            print(f"Saved {fname} -> {dest}")
    except Exception as e:
        print(f"Upload note: {e}")

# Re-check
train_file = find_file("train.jsonl")
val_file = find_file("val.jsonl")

print("\n" + "=" * 50)
print("  DATASET CONFIGURATION STATUS")
print("=" * 50)
print(f"  Train File: {train_file}")
print(f"  Val File:   {val_file}")

if train_file and os.path.exists(train_file):
    with open(train_file) as f:
        print(f"  -> Loaded {len(f.readlines())} training windows (Date-Held-Out)")
if val_file and os.path.exists(val_file):
    with open(val_file) as f:
        print(f"  -> Loaded {len(f.readlines())} validation windows")

## 🧠 Step 3: Load Base Model with Unsloth FastLanguageModel

In [ ]:
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template

MODEL_NAME = "unsloth/gemma-2-2b-it" # or "google/gemma-4-E2B-it" / "unsloth/Qwen2.5-14B-Instruct"
MAX_SEQ_LENGTH = 8192              # 8k context for long SQLite dumps
LOAD_IN_4BIT = False               # False for full bfloat16 on H100/A100; True for QLoRA on smaller GPUs

print(f"Loading {MODEL_NAME} with FastLanguageModel...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,
)

tokenizer = get_chat_template(tokenizer, chat_template="gemma")
print("Base model and tokenizer loaded successfully!")

## 💉 Step 4: Inject LoRA Target Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=32,                # LoRA Rank
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=64,       # LoRA Alpha scaling
    lora_dropout=0.0,    # 0.0 optimized for Unsloth
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

model.print_trainable_parameters()

## 📊 Step 5: Format Datasets with Chat Template

In [ ]:
import json
from datasets import Dataset

def prepare_dataset(filepath, tokenizer):
    samples = []
    with open(filepath, "r", encoding="utf-8") as f:
        for line in f:
            if not line.strip():
                continue
            item = json.loads(line)
            msgs = item.get("messages", [])
            if len(msgs) >= 2:
                text = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)
                samples.append({"text": text})
    return Dataset.from_list(samples)

train_dataset = prepare_dataset(train_file, tokenizer)
val_dataset = prepare_dataset(val_file, tokenizer) if val_file and os.path.exists(val_file) else None

print(f"Formatted {len(train_dataset)} training samples.")
if val_dataset:
    print(f"Formatted {len(val_dataset)} validation samples.")

## 🚀 Step 6: Train Model with SFTTrainer

In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

OUTPUT_DIR = "outputs_forensic_gemma_2b"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,   # Effective batch size = 8
    warmup_ratio=0.05,
    num_train_epochs=3,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    bf16=torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    logging_steps=1,
    optim="adamw_8bit" if torch.cuda.is_available() else "adamw_torch",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
    save_strategy="epoch",
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,
    args=training_args,
)

print("Starting GPU Fine-Tuning...")
trainer_stats = trainer.train()
print(f"Training Complete! Runtime: {trainer_stats.metrics['train_runtime']:.2f}s")

## 💾 Step 7: Save Fine-Tuned LoRA Adapter

In [ ]:
final_adapter_dir = f"{OUTPUT_DIR}/final_adapter"
print(f"Saving LoRA adapter to {final_adapter_dir}...")
model.save_pretrained(final_adapter_dir)
tokenizer.save_pretrained(final_adapter_dir)
print("Saved adapter successfully!")

## 🏆 Step 8: Automated In-Notebook Benchmark Evaluation
*(Self-contained evaluation scoring Citation Precision, Absence Reasoning, and Hallucinations)*

In [ ]:
import re
from tabulate import tabulate

FastLanguageModel.for_inference(model)

if not val_file or not os.path.exists(val_file):
    print("Validation file not available. Skipping evaluation.")
else:
    print(f"Evaluating fine-tuned model on {val_file}...")
    total_cited = 0
    valid_cited = 0
    hallucinated_cited = 0
    absence_detected_count = 0
    results = []

    with open(val_file, "r", encoding="utf-8") as f:
        val_lines = [json.loads(l) for l in f if l.strip()]

    for idx, item in enumerate(val_lines):
        msgs = item.get("messages", [])
        sys_prompt = msgs[0]["content"] if len(msgs) > 0 and msgs[0]["role"] == "system" else ""
        user_prompt = msgs[1]["content"] if len(msgs) > 1 and msgs[1]["role"] == "user" else ""
        target_answer = msgs[2]["content"] if len(msgs) > 2 and msgs[2]["role"] == "assistant" else ""

        valid_prompt_eids = set(re.findall(r"EVT-([a-f0-9]+)", user_prompt, flags=re.IGNORECASE))

        # Generate inference
        inp_msgs = [{"role": "system", "content": sys_prompt}, {"role": "user", "content": user_prompt}]
        inputs = tokenizer.apply_chat_template(inp_msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
        with torch.inference_mode():
            outputs = model.generate(inputs, max_new_tokens=300, temperature=0.1, use_cache=True)
        pred_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

        pred_eids = re.findall(r"EVT-([a-f0-9]+)", pred_text, flags=re.IGNORECASE)
        v_c = [e for e in pred_eids if e in valid_prompt_eids]
        h_c = [e for e in pred_eids if e not in valid_prompt_eids]

        total_cited += len(pred_eids)
        valid_cited += len(v_c)
        hallucinated_cited += len(h_c)

        has_absence = any(kw in pred_text.lower() for kw in ["without direct artifact support", "sqlcipher", "encrypted", "unrecoverable"])
        if has_absence:
            absence_detected_count += 1

        prec = (len(v_c) / len(pred_eids)) if pred_eids else 1.0
        results.append([f"Window #{idx+1}", len(pred_eids), len(v_c), f"{prec*100:.1f}%", "✅" if has_absence else "-"])

    precision = (valid_cited / total_cited * 100) if total_cited > 0 else 100.0
    print("\n" + "=" * 65)
    print("  FINAL FORENSIC EVALUATION SCORECARD")
    print("=" * 65)
    print(f"  Total Windows Evaluated:     {len(val_lines)}")
    print(f"  Overall Citation Precision:  {precision:.2f}%")
    print(f"  Hallucinated Phantom IDs:    {hallucinated_cited}")
    print(f"  Absence Auditing Accuracy:   {(absence_detected_count / len(val_lines) * 100):.1f}%")
    print("=" * 65)
    print(tabulate(results[:10], headers=["Window", "Total Cited", "Valid", "Precision", "Absence"], tablefmt="grid"))

## 🔍 Step 9: Live Interactive Reconstruction Test

In [ ]:
# Test sample prompt
test_prompt = """Date: 2019-03-15
EXTRACTED ARTIFACTS:
[EVT-a9668b712bee] 17:33 Twitter tweet_seen_or_posted <224423919>
[EVT-b86eea5bdfcc] 08:03 Twitter tweet_seen_or_posted <804341497441255424>
PROVENANCE:
EVT-a9668b712bee -> data\\com.twitter.android\\databases\\1068228364824178689-58.db :: statuses :: row 135
EVT-b86eea5bdfcc -> data\\com.twitter.android\\databases\\1068228364824178689-58.db :: statuses :: row 488

Reconstruct the user activity for this window."""

messages = [
    {"role": "system", "content": "You are a digital forensic analyst. Reconstruct the chronological user activity from the extracted Android artifacts. Cite evidence using exact [EVT-xxxx] IDs. Explicitly state unrecoverable apps."},
    {"role": "user", "content": test_prompt},
]

inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda" if torch.cuda.is_available() else "cpu")
with torch.inference_mode():
    outputs = model.generate(inputs, max_new_tokens=300, temperature=0.1, use_cache=True)

print("=== RECONSTRUCTED OUTPUT ===")
print(tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True))